# Fairness Analysis

This notebook:
1. Validates demographic distribution (prerequisite for meaningful fairness eval)
2. Compares fairness metrics: standard KG2RAG vs fairness-aware KG2RAG
3. Shows per-group accuracy breakdowns
4. Visualizes the accuracy-fairness tradeoff

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from fair_kg_rag.utils.io_utils import read_json

plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

metrics_dir = ROOT / 'outputs' / 'metrics'
processed_dir = ROOT / 'data' / 'processed'
split = 'dev'

## 1. Demographic Distribution Check

Before interpreting fairness metrics, verify the dataset has enough demographic diversity.

In [ ]:
demo_path = processed_dir / 'entity_demographics.json'
if demo_path.exists():
    demographics = read_json(demo_path)
    demo_df = pd.DataFrame(demographics)
    
    print(f'Total entities with demographics: {len(demo_df)}')
    print(f'\n--- Gender ---')
    gender_counts = demo_df['gender'].value_counts(dropna=False)
    print(gender_counts)
    gender_valid = demo_df[demo_df['gender'].notna()]
    if len(gender_valid) > 0:
        ratio = gender_valid['gender'].value_counts(normalize=True).iloc[0]
        print(f'Max group ratio: {ratio:.2%} — {"OK" if ratio < 0.85 else "SKEWED"}')
    
    print(f'\n--- Geographic Group ---')
    geo_counts = demo_df['geo_group'].value_counts()
    print(geo_counts)
    geo_valid = demo_df[demo_df['geo_group'] != 'unknown']
    if len(geo_valid) > 0:
        ratio = geo_valid['geo_group'].value_counts(normalize=True).iloc[0]
        print(f'Max group ratio: {ratio:.2%} — {"OK" if ratio < 0.85 else "SKEWED"}')
else:
    print(f'Missing {demo_path}. Run: python scripts/fetch_demographics.py')

## 2. Load Experiment Metrics

In [ ]:
experiments = {
    'baseline_naive': 'Naive RAG',
    'kg2rag_standard': 'KG2RAG (standard)',
    'kg2rag_fair': 'KG2RAG (fair)',
}

all_metrics = {}
for exp_key, exp_label in experiments.items():
    path = metrics_dir / f'{split}_{exp_key}_metrics.json'
    if path.exists():
        all_metrics[exp_label] = read_json(path)
        print(f'Loaded: {exp_label}')
    else:
        print(f'Missing: {path}')

if not all_metrics:
    print('\nNo experiment metrics found yet. Run experiments first:')
    print('  python scripts/run_experiment.py --experiment kg2rag_standard')
    print('  python scripts/run_experiment.py --experiment kg2rag_fair')

## 3. Fairness Metrics Comparison

In [ ]:
if all_metrics:
    rows = []
    for exp_label, metrics in all_metrics.items():
        acc = metrics.get('accuracy', {})
        fairness = metrics.get('fairness', {})
        
        row = {
            'Experiment': exp_label,
            'EM': acc.get('exact_match', 0),
            'F1': acc.get('answer_f1', 0),
        }
        for attr, attr_metrics in fairness.items():
            if isinstance(attr_metrics, dict):
                row[f'{attr}_parity'] = attr_metrics.get('demographic_parity', 0)
                row[f'{attr}_tpr_gap'] = attr_metrics.get('tpr_gap', 0)
        rows.append(row)
    
    results_df = pd.DataFrame(rows).set_index('Experiment')
    print('=== Accuracy + Fairness Summary ===')
    display(results_df.round(4))
else:
    print('No metrics to display.')

In [ ]:
if all_metrics and len(all_metrics) >= 2:
    # Fairness comparison bar chart
    fairness_cols = [c for c in results_df.columns if 'parity' in c or 'gap' in c]
    if fairness_cols:
        fig, ax = plt.subplots(figsize=(10, 5))
        results_df[fairness_cols].plot.bar(ax=ax)
        ax.set_title('Fairness Metrics: Lower = More Fair')
        ax.set_ylabel('Disparity')
        ax.set_xlabel('')
        ax.axhline(y=0, color='black', linewidth=0.5)
        ax.legend(loc='upper right')
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()

## 4. Per-Group Accuracy Breakdown

In [ ]:
if all_metrics:
    for exp_label, metrics in all_metrics.items():
        fairness = metrics.get('fairness', {})
        if not fairness:
            continue
        print(f'\n=== {exp_label} ===')
        for attr, attr_metrics in fairness.items():
            if not isinstance(attr_metrics, dict):
                continue
            print(f'\n  {attr}:')
            # Extract per-group accuracy keys
            group_accs = {k: v for k, v in attr_metrics.items() if k.endswith('_accuracy')}
            group_counts = {k: v for k, v in attr_metrics.items() if k.endswith('_count')}
            for k, v in sorted(group_accs.items()):
                group_name = k.replace('_accuracy', '')
                count = group_counts.get(f'{group_name}_count', '?')
                print(f'    {group_name}: accuracy={v:.4f} (n={count})')
            dp = attr_metrics.get('demographic_parity', 0)
            print(f'    --> Demographic Parity Gap: {dp:.4f}')

## 5. Accuracy vs Fairness Tradeoff

In [ ]:
if all_metrics and len(all_metrics) >= 2:
    # Scatter: x = overall F1, y = max demographic parity gap
    scatter_data = []
    for exp_label, metrics in all_metrics.items():
        f1 = metrics.get('accuracy', {}).get('answer_f1', 0)
        max_parity = 0
        for attr, am in metrics.get('fairness', {}).items():
            if isinstance(am, dict):
                max_parity = max(max_parity, am.get('demographic_parity', 0))
        scatter_data.append({'Experiment': exp_label, 'F1': f1, 'Max Parity Gap': max_parity})
    
    sdf = pd.DataFrame(scatter_data)
    
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(sdf['F1'], sdf['Max Parity Gap'], s=100, zorder=5)
    for _, row in sdf.iterrows():
        ax.annotate(row['Experiment'], (row['F1'], row['Max Parity Gap']),
                    textcoords='offset points', xytext=(8, 4), fontsize=9)
    ax.set_xlabel('Answer F1 (higher = better accuracy)')
    ax.set_ylabel('Max Demographic Parity Gap (lower = more fair)')
    ax.set_title('Accuracy vs Fairness Tradeoff')
    ax.axhline(y=0.05, color='green', linestyle='--', alpha=0.5, label='Fair threshold')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Need at least 2 experiments for tradeoff plot.')